# Business Entity Resolution — full pipeline
Settings (right panel): **Accelerator = GPU T4 x2**, **Internet = On**. Inputs: the data dataset and the code dataset.

Run with **Save Version → Save & Run All (Commit)** so it keeps going with the browser closed (~4–5 h).

In [ ]:
# 1. setup: install the two missing packages, find data + code folders automatically
!pip install -q rapidfuzz anyascii
import glob, os
DATA = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/train/train_ground_truth.tsv', recursive=True)[0]))
CODE = os.path.dirname(os.path.dirname(glob.glob('/kaggle/input/**/src/prep.py', recursive=True)[0]))
print('DATA =', DATA, os.listdir(DATA)); print('CODE =', CODE, os.listdir(CODE))
%cd /kaggle/working
!nvidia-smi -L

In [ ]:
# 2. normalize all records + learn transliteration maps (CPU, ~30 min)
!python {CODE}/src/prep.py --data {DATA} --work work

In [ ]:
# 3. fine-tune multilingual-e5-small (GPU, ~15 min)
!python {CODE}/src/finetune.py --work work

In [ ]:
# 4. candidate generation on train — prints BLOCKING RECALL (send me this)
!python {CODE}/src/block.py --work work --split train

In [ ]:
# 5. train matcher — prints VALIDATION F0.5 (send me this)
!python {CODE}/src/train.py --work work

In [ ]:
# 6. candidate generation on test
!python {CODE}/src/block.py --work work --split test

In [ ]:
# 7. score test + write output/matching_results.tsv and output/candidate_pairs.tsv
!python {CODE}/src/predict.py --work work --out output

In [ ]:
# 8. record exact package versions, show results
!pip freeze > output/requirements_kaggle.txt
!cp work/model_cfg.json work/recall.json work/val_errors.tsv output/
!ls -lh output && head -3 output/matching_results.tsv